# LLM Judge Pipeline —— ChatDev 多智能体轨迹评估

本 Notebook 自动发现 `dataset_mini/` 下所有工作区日志，
通过 GPT-4o 评估每条轨迹的失败模式（failure modes），
并将格式化结果写入 `judge_results/`。

## 1. 环境初始化

In [17]:
import os
import re
import json
import glob
import time
from pathlib import Path

import dotenv
dotenv.load_dotenv(override=True)

from openai import OpenAI

# ---- 配置 ----
MODEL = "gpt-4o"
DATASET_DIR = Path("dataset_mini")
RESULTS_DIR = Path("judge_results_3")
RESULTS_DIR.mkdir(exist_ok=True)

base_url = os.environ.get("BASE_URL")
openai_api_key = os.environ.get("OPENAI_API_KEY")

client = OpenAI(
    api_key=openai_api_key,
    base_url=base_url,
)

print(f"Model: {MODEL}")
print(f"Dataset dir: {DATASET_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

Model: gpt-4o
Dataset dir: D:\Works\code\winter-like-ai\ChatDev\dataset_mini
Results dir: D:\Works\code\winter-like-ai\ChatDev\judge_results_3


## 2. 加载 Taxonomy（定义 + 示例）

In [18]:
definitions = open("taxonomy_definitions_examples/definitions.txt", "r").read()
examples    = open("taxonomy_definitions_examples/examples.txt", "r").read()

# 失败模式代码及其名称
FAILURE_MODE_NAMES = {
    '1.1': 'Disobey Task Specification',
    '1.2': 'Disobey Role Specification',
    '1.3': 'Step Repetition',
    '1.4': 'Loss of Conversation History',
    '1.5': 'Unaware of Termination Conditions',
    '2.1': 'Conversation Reset',
    '2.2': 'Fail to Ask for Clarification',
    '2.3': 'Task Derailment',
    '2.4': 'Information Withholding',
    '2.5': 'Ignored Other Agent Input',
    '2.6': 'Action-Reasoning Mismatch',
    '3.1': 'Premature Termination',
    '3.2': 'Weak Verification',
    '3.3': 'No or Incorrect Verification',
}
FAILURE_MODES = list(FAILURE_MODE_NAMES.keys())
print(f"Loaded {len(definitions)} chars of definitions, {len(examples)} chars of examples")
print(f"Tracking {len(FAILURE_MODES)} failure modes")

Loaded 13162 chars of definitions, 67469 chars of examples
Tracking 14 failure modes


## 3. 自动发现 dataset_mini 下的日志轨迹

In [19]:
# 每个子目录下有唯一一个 .log 文件
log_entries = []
for sub in sorted(DATASET_DIR.iterdir()):
    if not sub.is_dir():
        continue
    logs = list(sub.glob("*.log"))
    if len(logs) == 1:
        log_entries.append({
            "dir_name": sub.name,
            "log_path": logs[0],
        })
    elif len(logs) > 1:
        print(f"WARNING: {sub.name} has {len(logs)} log files, using the first one")
        log_entries.append({
            "dir_name": sub.name,
            "log_path": logs[0],
        })
    else:
        print(f"WARNING: {sub.name} has no .log files, skipping")

print(f"\nDiscovered {len(log_entries)} trace entries:")
for i, entry in enumerate(log_entries):
    size_kb = entry['log_path'].stat().st_size / 1024
    print(f"  [{i+1:2d}] {entry['dir_name']}  ({size_kb:.0f} KB)")


Discovered 15 trace entries:
  [ 1] Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331212914  (399 KB)
  [ 2] Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331215105  (356 KB)
  [ 3] Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331220035  (268 KB)
  [ 4] CLI_Text_File_Word_Counter_DefaultOrganization_20260331200439  (248 KB)
  [ 5] CLI_Text_File_Word_Counter_DefaultOrganization_20260331210834  (257 KB)
  [ 6] CLI_Text_File_Word_Counter_DefaultOrganization_20260331214035  (250 KB)
  [ 7] CLI_Unit_Converter_Temperature_DefaultOrganization_20260331212438  (255 KB)
  [ 8] CLI_Unit_Converter_Temperature_DefaultOrganization_20260331215600  (256 KB)
  [ 9] CLI_Unit_Converter_Temperature_DefaultOrganization_20260331215959  (262 KB)
  [10] Directory_Tree_Generator_CLI_DefaultOrganization_20260331200919  (326 KB)
  [11] Directory_Tree_Generator_CLI_DefaultOrganization_20260331211344  (322 KB)
  [12] Directory_Tree_Generator_CLI_DefaultOrganization_20260331214549  (293 KB)
 

## 4. LLM 评估器

In [20]:
def chat_completion(prompt: str) -> str:
    """调用 OpenAI API 获取评估结果。"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
    )
    return response.choices[0].message.content if response.choices else None


def build_evaluation_prompt(trace: str) -> str:
    """构建完整的评估 prompt。"""
    prompt = (
        "Below I will provide a multiagent system trace. provide me an analysis of the failure modes and inefficiencies as I will say below. \n"
        "In the traces, analyze the system behaviour."
        "There are several failure modes in multiagent systems I identified. I will provide them below. Tell me if you encounter any of them, as a binary yes or no. \n"
        "Also, give me a one sentence (be brief) summary of the problems with the inefficiencies or failure modes in the trace. Only mark a failure mode if you can provide an example of it in the trace, and specify that in your summary at the end"
        "Also tell me whether the task is successfully completed or not, as a binary yes or no."
        "At the very end, I provide you with the definitions of the failure modes and inefficiencies. After the definitions, I will provide you with examples of the failure modes and inefficiencies for you to understand them better."
        "Tell me if you encounter any of them between the @@ symbols as I will say below, as a binary yes or no."
        "Here are the things you should answer. Start after the @@ sign and end before the next @@ sign (do not include the @@ symbols in your answer):"
        "*** begin of things you should answer *** @@"
        "A. Freeform text summary of the problems with the inefficiencies or failure modes in the trace: <summary>"
        "B. Whether the task is successfully completed or not: <yes or no>"
        "C. Whether you encounter any of the failure modes or inefficiencies:"
        "1.1 Disobey Task Specification: <yes or no>"
        "1.2 Disobey Role Specification: <yes or no>"
        "1.3 Step Repetition: <yes or no>"
        "1.4 Loss of Conversation History: <yes or no>"
        "1.5 Unaware of Termination Conditions: <yes or no>"
        "2.1 Conversation Reset: <yes or no>"
        "2.2 Fail to Ask for Clarification: <yes or no>"
        "2.3 Task Derailment: <yes or no>"
        "2.4 Information Withholding: <yes or no>"
        "2.5 Ignored Other Agent's Input: <yes or no>"
        "2.6 Action-Reasoning Mismatch: <yes or no>"
        "3.1 Premature Termination: <yes or no>"
        "3.2 Weak Verification: <yes or no>"
        "3.3 No or Incorrect Verification: <yes or no>"
        "@@*** end of your answer ***"
        "An example answer is: \n"
        "A. The task is not completed due to disobeying role specification as agents went rogue and started to chat with each other instead of completing the task. Agents derailed and verifier is not strong enough to detect it.\n"
        "B. no \n"
        "C. \n"
        "1.1 no \n"
        "1.2 no \n"
        "1.3 no \n"
        "1.4 no \n"
        "1.5 no \n"
        "2.1 no \n"
        "2.2 no \n"
        "2.3 yes \n"
        "2.4 no \n"
        "2.5 no \n"
        "2.6 yes \n"
        "3.1 no \n"
        "3.2 yes \n"
        "3.3 no \n"
        "Here is the trace: \n"
        f"{trace}"
        "Also, here are the explanations (definitions) of the failure modes and inefficiencies: \n"
        f"{definitions} \n"
        "Here are some examples of the failure modes and inefficiencies: \n"
        f"{examples}"
    )
    return prompt


# Token 限制：prompt + examples 不超过 ~1M chars
MAX_TRACE_CHARS = 1_048_570 - len(examples) - len(definitions) - 2000  # 留余量
print(f"Max trace length: {MAX_TRACE_CHARS:,} chars")

Max trace length: 965,939 chars


## 5. 解析 LLM 返回结果

In [21]:
def parse_single_response(response: str) -> dict:
    """
    解析单条 LLM 评估结果，返回结构化字典：
    {
        'summary': str,            # A. 自由文本摘要
        'task_completed': bool,     # B. 任务是否完成
        'failure_modes': {          # C. 各失败模式 yes/no -> True/False
            '1.1': True/False,
            ...
        },
        'raw_response': str         # 原始返回
    }
    """
    result = {
        'summary': '',
        'task_completed': None,
        'failure_modes': {},
        'raw_response': response,
    }
    
    if not response:
        for mode in FAILURE_MODES:
            result['failure_modes'][mode] = False
        return result
    
    cleaned = response.strip()
    # 移除 @@ 标记
    cleaned = cleaned.replace('@@', '')
    
    # ---- 解析 A. Summary ----
    summary_match = re.search(r'A\.\s*(.*?)(?=\nB\.)', cleaned, re.DOTALL)
    if summary_match:
        result['summary'] = summary_match.group(1).strip()
    
    # ---- 解析 B. Task completed ----
    task_match = re.search(r'B\.\s*(yes|no)', cleaned, re.IGNORECASE)
    if task_match:
        result['task_completed'] = task_match.group(1).lower() == 'yes'
    
    # ---- 解析 C. 各失败模式 ----
    for mode in FAILURE_MODES:
        # 尝试多种匹配模式
        patterns = [
            rf"{re.escape(mode)}\s*[:\-]?\s*(yes|no)",
            rf"C\..*?{re.escape(mode)}.*?(yes|no)",
        ]
        found = False
        for pattern in patterns:
            match = re.search(pattern, cleaned, re.IGNORECASE | re.DOTALL)
            if match:
                result['failure_modes'][mode] = match.group(1).lower() == 'yes'
                found = True
                break
        if not found:
            result['failure_modes'][mode] = False  # 默认 no
    
    return result


# 验证解析器
test_response = """A. The system had step repetition in the code review phase and weak verification where the reviewer missed issues.
B. yes
C.
1.1 no
1.2 no
1.3 yes
1.4 no
1.5 no
2.1 no
2.2 no
2.3 no
2.4 no
2.5 no
2.6 no
3.1 no
3.2 yes
3.3 no"""
parsed = parse_single_response(test_response)
print("Parser test:")
print(f"  Summary: {parsed['summary'][:60]}...")
print(f"  Task completed: {parsed['task_completed']}")
print(f"  Failure modes detected: {[m for m, v in parsed['failure_modes'].items() if v]}")

Parser test:
  Summary: The system had step repetition in the code review phase and ...
  Task completed: True
  Failure modes detected: ['1.3', '3.2']


## 6. 批量评估 15 条轨迹

In [22]:
all_results = []

for i, entry in enumerate(log_entries):
    dir_name = entry['dir_name']
    log_path = entry['log_path']
    result_file = RESULTS_DIR / f"{dir_name}.json"
    
    # 跳过已评估的条目
    if result_file.exists():
        print(f"[{i+1:2d}/{len(log_entries)}] SKIP (already evaluated): {dir_name}")
        with open(result_file, 'r', encoding='utf-8') as f:
            all_results.append(json.load(f))
        continue
    
    print(f"[{i+1:2d}/{len(log_entries)}] Evaluating: {dir_name}")
    
    # 读取日志
    trace = log_path.read_text(encoding='utf-8', errors='replace')
    original_len = len(trace)
    
    # 截断过长的 trace
    if len(trace) > MAX_TRACE_CHARS:
        trace = trace[:MAX_TRACE_CHARS]
        print(f"  ⚠ Trace truncated: {original_len:,} -> {MAX_TRACE_CHARS:,} chars")
    
    # 调用 LLM
    try:
        t0 = time.time()
        prompt = build_evaluation_prompt(trace)
        raw_response = chat_completion(prompt)
        elapsed = time.time() - t0
        print(f"  ✓ Got response in {elapsed:.1f}s ({len(raw_response)} chars)")
    except Exception as e:
        print(f"  ✗ API error: {e}")
        raw_response = None
        elapsed = 0
    
    # 解析
    parsed = parse_single_response(raw_response)
    
    # 构建结果
    result_entry = {
        'dir_name': dir_name,
        'log_file': log_path.name,
        'trace_chars': original_len,
        'truncated': original_len > MAX_TRACE_CHARS,
        'evaluation_time_sec': round(elapsed, 1),
        'summary': parsed['summary'],
        'task_completed': parsed['task_completed'],
        'failure_modes': parsed['failure_modes'],
        'failure_modes_detected': [m for m, v in parsed['failure_modes'].items() if v],
        'raw_response': parsed['raw_response'],
    }
    
    # 保存单条结果
    with open(result_file, 'w', encoding='utf-8') as f:
        json.dump(result_entry, f, ensure_ascii=False, indent=2)
    
    all_results.append(result_entry)
    
    # 打印简要结果
    detected = result_entry['failure_modes_detected']
    status = '✅' if result_entry['task_completed'] else '❌'
    print(f"  {status} Task completed: {result_entry['task_completed']}")
    if detected:
        print(f"  🔍 Failures: {', '.join(detected)}")
    else:
        print(f"  🔍 No failure modes detected")
    print()

print(f"\n{'='*60}")
print(f"Evaluation complete: {len(all_results)}/{len(log_entries)} entries processed")

[ 1/15] Evaluating: Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331212914
  ✓ Got response in 6.8s (700 chars)
  ❌ Task completed: None
  🔍 No failure modes detected

[ 2/15] Evaluating: Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331215105
  ✓ Got response in 5.5s (385 chars)
  ✅ Task completed: True
  🔍 Failures: 2.2, 3.2

[ 3/15] Evaluating: Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331220035
  ✓ Got response in 8.5s (500 chars)
  ✅ Task completed: True
  🔍 No failure modes detected

[ 4/15] Evaluating: CLI_Text_File_Word_Counter_DefaultOrganization_20260331200439
  ✓ Got response in 5.5s (485 chars)
  ✅ Task completed: True
  🔍 Failures: 1.1, 2.3, 3.2

[ 5/15] Evaluating: CLI_Text_File_Word_Counter_DefaultOrganization_20260331210834
  ✓ Got response in 9.5s (751 chars)
  ❌ Task completed: None
  🔍 Failures: 1.3, 3.2

[ 6/15] Evaluating: CLI_Text_File_Word_Counter_DefaultOrganization_20260331214035
  ✓ Got response in 7.5s (460 chars)
  ✅ Task completed:

## 7. 汇总统计并保存

In [23]:
# ---- 汇总统计 ----
total = len(all_results)
task_completed_count = sum(1 for r in all_results if r['task_completed'])

# 按任务分组进行分析（每个任务有 3 条记录）
task_groups = {}
for r in all_results:
    # 从目录名提取任务名（去掉 DefaultOrganization_时间戳 后缀）
    parts = r['dir_name'].rsplit('_DefaultOrganization_', 1)
    task_name = parts[0] if len(parts) == 2 else r['dir_name']
    if task_name not in task_groups:
        task_groups[task_name] = []
    task_groups[task_name].append(r)

# ---- 失败模式频率统计 ----
fm_counts = {mode: 0 for mode in FAILURE_MODES}
for r in all_results:
    for mode in FAILURE_MODES:
        if r['failure_modes'].get(mode, False):
            fm_counts[mode] += 1

# ---- 显示汇总 ----
print(f"Total entries: {total}")
print(f"Task completed: {task_completed_count}/{total} ({task_completed_count/total*100:.1f}%)")
print(f"Task groups: {len(task_groups)}")
print()

print("--- Failure Mode Frequency ---")
print(f"{'Code':<6} {'Name':<40} {'Count':>5} {'Rate':>7}")
print("-" * 62)
for mode in FAILURE_MODES:
    name = FAILURE_MODE_NAMES[mode]
    count = fm_counts[mode]
    rate = count / total * 100 if total > 0 else 0
    bar = '█' * int(rate / 5)  # 简单柱状图
    print(f"{mode:<6} {name:<40} {count:>5} {rate:>6.1f}% {bar}")

print()
print("--- Per-Task Summary ---")
for task_name, entries in task_groups.items():
    completed = sum(1 for e in entries if e['task_completed'])
    print(f"\n📋 {task_name} ({completed}/{len(entries)} completed)")
    for e in entries:
        status = '✅' if e['task_completed'] else '❌'
        detected = e['failure_modes_detected']
        det_str = ', '.join(detected) if detected else 'none'
        print(f"   {status} {e['log_file'][:50]}...  failures=[{det_str}]")

Total entries: 15
Task completed: 11/15 (73.3%)
Task groups: 5

--- Failure Mode Frequency ---
Code   Name                                     Count    Rate
--------------------------------------------------------------
1.1    Disobey Task Specification                   2   13.3% ██
1.2    Disobey Role Specification                   0    0.0% 
1.3    Step Repetition                              2   13.3% ██
1.4    Loss of Conversation History                 0    0.0% 
1.5    Unaware of Termination Conditions            0    0.0% 
2.1    Conversation Reset                           0    0.0% 
2.2    Fail to Ask for Clarification                1    6.7% █
2.3    Task Derailment                              2   13.3% ██
2.4    Information Withholding                      0    0.0% 
2.5    Ignored Other Agent Input                    0    0.0% 
2.6    Action-Reasoning Mismatch                    1    6.7% █
3.1    Premature Termination                        1    6.7% █
3.2    Weak Ver

In [24]:
# ---- 保存汇总报告 ----
summary_report = {
    'meta': {
        'model': MODEL,
        'total_entries': total,
        'task_completed_count': task_completed_count,
        'task_completed_rate': round(task_completed_count / total * 100, 1) if total > 0 else 0,
        'num_task_groups': len(task_groups),
    },
    'failure_mode_frequency': {
        mode: {
            'name': FAILURE_MODE_NAMES[mode],
            'count': fm_counts[mode],
            'rate_percent': round(fm_counts[mode] / total * 100, 1) if total > 0 else 0,
        }
        for mode in FAILURE_MODES
    },
    'per_task_summary': {
        task_name: {
            'num_runs': len(entries),
            'completed_count': sum(1 for e in entries if e['task_completed']),
            'runs': [
                {
                    'dir_name': e['dir_name'],
                    'task_completed': e['task_completed'],
                    'summary': e['summary'],
                    'failure_modes_detected': e['failure_modes_detected'],
                }
                for e in entries
            ],
        }
        for task_name, entries in task_groups.items()
    },
}

summary_file = RESULTS_DIR / "_summary.json"
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(summary_report, f, ensure_ascii=False, indent=2)

print(f"\n✅ Summary saved to: {summary_file.resolve()}")
print(f"✅ Individual results saved to: {RESULTS_DIR.resolve()}")
print(f"\nFiles in judge_results/:")
for p in sorted(RESULTS_DIR.iterdir()):
    print(f"  {p.name} ({p.stat().st_size / 1024:.1f} KB)")


✅ Summary saved to: D:\Works\code\winter-like-ai\ChatDev\judge_results_3\_summary.json
✅ Individual results saved to: D:\Works\code\winter-like-ai\ChatDev\judge_results_3

Files in judge_results/:
  _summary.json (10.1 KB)
  Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331212914.json (1.3 KB)
  Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331215105.json (1.2 KB)
  Basic_Network_Ping_Tool_CLI_DefaultOrganization_20260331220035.json (1.4 KB)
  CLI_Text_File_Word_Counter_DefaultOrganization_20260331200439.json (1.4 KB)
  CLI_Text_File_Word_Counter_DefaultOrganization_20260331210834.json (1.9 KB)
  CLI_Text_File_Word_Counter_DefaultOrganization_20260331214035.json (1.4 KB)
  CLI_Unit_Converter_Temperature_DefaultOrganization_20260331212438.json (1.4 KB)
  CLI_Unit_Converter_Temperature_DefaultOrganization_20260331215600.json (1.9 KB)
  CLI_Unit_Converter_Temperature_DefaultOrganization_20260331215959.json (1.4 KB)
  Directory_Tree_Generator_CLI_DefaultOrganization_20260331